# Datos categóricos {#sec-categorical-data}

## Introducción

En este capítulo, veremos cómo trabajar con variables categóricas, es decir, variables que tienen un conjunto fijo y conocido de valores posibles. Este capítulo le debe muchísimo a la [documentación](https://pandas.pydata.org/) de **pandas**.


In [ ]:
# remove cell
import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline

# Plot settings
plt.style.use("https://github.com/aeturrell/python4DS/raw/main/plot_style.txt")
matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

### Requisitos previos

Este capítulo usará el paquete de análisis de datos **pandas**.

## El tipo de dato category

Todo en Python tiene un tipo, incluso los datos de las columnas de un dataframe de **pandas**. Aunque quizá estés más familiarizado con los números e incluso con los strings, también existe un tipo de dato especial para datos categóricos llamado `Categorical`. Usar variables categóricas (cuando corresponde) tiene algunas ventajas:

- permiten llevar la cuenta incluso cuando los elementos de una categoría no están presentes, lo que a veces puede ser tan interesante como cuando sí lo están (imagina que descubres que nadie de una escuela en particular va a la universidad)
- pueden usar muchísima menos memoria de tu computadora que codificar la misma información de otras maneras
- pueden usarse de forma eficiente con paquetes de modelado, donde se reconocerán como posibles 'variables dummy', o con paquetes de gráficos, que las tratarán como valores discretos
- puedes ordenarlas (por ejemplo, "neutral", "de acuerdo", "muy de acuerdo")

Todos los valores de datos categóricos de una columna de **pandas** están en las categorías dadas o toman el valor `np.nan`. 

## Crear datos categóricos

Creemos una columna de datos categórica:

In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame({"A": ["a", "b", "c", "a"]})

df["A"] = df["A"].astype("category")
df["A"]

Observa que obtenemos información adicional al final de la serie mostrada: se nos indica no solo que se trata de una columna de tipo categórico, sino también que tiene tres valores: 'a', 'b' y 'c'.

También puedes usar funciones especiales, como `pd.cut()`, para agrupar datos en intervalos discretos. Aquí tienes un ejemplo en el que especificamos directamente las etiquetas de las categorías:


In [ ]:
df = pd.DataFrame({"value": np.random.randint(0, 100, 20)})
labels = [f"{i} - {i+9}" for i in range(0, 100, 10)]
df["group"] = pd.cut(df.value, range(0, 105, 10), right=False, labels=labels)
df.head()

En el ejemplo anterior, la columna `group` es de tipo categórico.

Otra forma de crear una variable categórica es usar directamente la función `pd.Categorical()`:

In [ ]:
raw_cat = pd.Categorical(
    ["a", "b", "c", "a", "d", "a", "c"], categories=["b", "c", "d"]
)
raw_cat

Luego podemos introducirla en un dataframe:

In [ ]:
df = pd.DataFrame(raw_cat, columns=["cat_type"])
df["cat_type"]

Observa que aparecen NaN para cualquier valor que *no* esté en las categorías que especificamos; puedes encontrar más sobre esto en @sec-missing-values.

También puedes crear categorías ordenadas:

In [ ]:
ordered_cat = pd.Categorical(
    ["a", "b", "c", "a", "d", "a", "c"],
    categories=["a", "b", "c", "d"],
    ordered=True,
)
ordered_cat

## Trabajar con categorías

Los datos categóricos tienen las propiedades `categories` y `ordered`, que indican, respectivamente, los valores posibles y si el orden importa o no. Estas propiedades se exponen como `.cat.categories` y `.cat.ordered`. Si no especificas manualmente las categorías y el orden, se infieren a partir de los argumentos pasados.

Veamos algunos ejemplos:

In [ ]:
df["cat_type"].cat.categories

In [ ]:
df["cat_type"].cat.ordered

Si los datos categóricos están ordenados (es decir, `.cat.ordered == True`), el orden de las categorías tiene significado y se pueden realizar ciertas operaciones: puedes ordenar valores (con `.sort_values`) y aplicar `.min` y `.max`.

### Renombrar categorías

Las categorías se renombran mediante el método `rename_categories()` (que funciona con una lista o un diccionario).

In [ ]:
df["cat_type"] = df["cat_type"].cat.rename_categories(["alpha", "beta", "gamma"])

Con bastante frecuencia te encontrarás en una situación en la que quieras añadir una categoría. Puedes hacerlo con `.add_categories()`:

In [ ]:
df["cat_type"] = df["cat_type"].cat.add_categories(["delta"])
df["cat_type"]

De forma similar, existen las funciones `.remove_categories()` y `.remove_unused_categories()`. `.set_categories` añade y elimina categorías de una sola vez. Una de las propiedades útiles de set categories es que  Recuerda, eso sí, que necesitas usar `df["columnname"].cat` antes de llamar a cualquier función cat(egory).

## Operaciones con categorías

Como ya mencionamos, las categorías ordenadas admiten de por sí algunas operaciones. Pero hay otras que funcionan con cualquier conjunto de categorías. Quizá la más útil sea `value_counts()`

In [ ]:
df["cat_type"].value_counts()

Observa que, aunque 'delta' no aparece en absoluto, obtiene un conteo (de cero). Este seguimiento de los valores ausentes puede ser muy práctico.

`mode()` es otra:

In [ ]:
df["cat_type"].mode()

Y si tu columna categórica resulta estar formada por *elementos* que admiten operaciones, esas mismas operaciones seguirán funcionando. Por ejemplo,

In [ ]:
time_df = pd.DataFrame(
    pd.Series(pd.date_range("2015/05/01", periods=5, freq="M"), dtype="category"),
    columns=["datetime"],
)
time_df

In [ ]:
time_df["datetime"].dt.month

Por último, si alguna vez necesitas convertir los valores reales de tu columna categórica en un código, puedes usar `.cat.codes` para obtener códigos únicos para cada valor.

In [ ]:
time_df["datetime"].cat.codes